# Edge-Enhanced GraphSAGE — Training Evidence (Google Colab)

**Purpose:** run the full 4-stage ablation training *in the cloud*, so the training
process and its outputs are visible and reproducible — not locked on a local PC.

| Stage | What it adds | Script |
|---|---|---|
| 1 | GraphSAGE baseline (BCE + pos_weight) | `scripts/train_baseline.py` |
| 2 | + Edge-MLP attention (**Novelty 1**) | `scripts/train_edge_mlp.py` |
| 3a | + Focal Loss, full-batch (ablation arm) | `scripts/train_focal.py` |
| 3b | + Graph-Aware Imbalance Sampler (**Novelty 2**) | `scripts/train_full.py` |

Runtime: ~15–40 min total on a T4 (Runtime → Change runtime type → **T4 GPU**).
Prerequisite: `paysim_graph.pt` in Drive at `MyDrive/R26-IT-121-data/` (built once by `scripts/build_graph.py`).


## 1. GPU evidence + environment


In [ ]:
import sys
IS_COLAB = 'google.colab' in sys.modules
print('Environment:', 'Google Colab' if IS_COLAB else 'Local')
!nvidia-smi


In [ ]:
from pathlib import Path

if IS_COLAB:
    if not Path('/content/R26-IT-121').exists():
        !git clone -q https://github.com/LEXES7/R26-IT-121.git /content/R26-IT-121
    %cd /content/R26-IT-121/GraphSage
    %pip install -q torch-geometric
    sys.path.insert(0, '/content/R26-IT-121/GraphSage/src')

    from google.colab import drive
    drive.mount('/content/drive')
    GRAPH_SRC = Path('/content/drive/MyDrive/R26-IT-121-data/paysim_graph.pt')
    GRAPH_DST = Path('data/graph/paysim_graph.pt')
    GRAPH_DST.parent.mkdir(parents=True, exist_ok=True)
    if not GRAPH_DST.exists():
        print('Copying graph from Drive (195 MB)...')
        import shutil; shutil.copy(GRAPH_SRC, GRAPH_DST)
print('Ready.')


## 2. Train all four stages

Each cell runs the *same script that lives in the repo* — the epoch logs below are
the training evidence. Early stopping keeps runs short.


In [ ]:
!python scripts/train_baseline.py


In [ ]:
!python scripts/train_edge_mlp.py


In [ ]:
!python scripts/train_focal.py


In [ ]:
!python scripts/train_full.py


## 3. Training curves (from the per-epoch history each script saves)


In [ ]:
import json
import matplotlib.pyplot as plt

RUNS = [
    ('reports/stage1_metrics.json',  'Stage 1 baseline',   '#9ec5f4'),
    ('reports/stage2_metrics.json',  'Stage 2 +Edge-MLP',  '#5598e7'),
    ('reports/stage3a_metrics.json', 'Stage 3a +Focal',    '#2a78d6'),
    ('reports/stage3_metrics.json',  'Stage 3b full',      '#104281'),
]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
fig.patch.set_facecolor('white')
for path, label, color in RUNS:
    hist = json.load(open(path))['history']
    epochs = range(1, len(hist) + 1)
    axes[0].plot(epochs, [h['train_loss'] for h in hist], color=color, lw=2, label=label)
    axes[1].plot(epochs, [h['val_f1'] for h in hist], color=color, lw=2, label=label)
axes[0].set_title('Training loss'); axes[1].set_title('Validation F1 (default threshold)')
for ax in axes:
    ax.set_xlabel('epoch'); ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', color='#e1e0d9', lw=0.7)
axes[1].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()


## 4. Honest ablation — threshold tuned on validation, frozen for test

Threshold selection is standard **decision-threshold tuning** for imbalanced
classification (cf. scikit-learn `TunedThresholdClassifierCV`; Elkan 2001): sweep
thresholds on the *validation* split, pick the F1-max operating point on the PR
curve, freeze it, then report *test* metrics at that frozen threshold. The test
set never influences the choice.


In [ ]:
!python scripts/eval_with_tuned_threshold.py


## 5. Export checkpoints + reports to Drive (timestamped evidence)


In [ ]:
if IS_COLAB:
    import shutil, time
    stamp = time.strftime('%Y%m%d_%H%M')
    out = Path(f'/content/drive/MyDrive/R26-IT-121-data/colab_runs/{stamp}')
    out.mkdir(parents=True, exist_ok=True)
    for f in list(Path('checkpoints').glob('*.pt')) + list(Path('reports').glob('*.json')):
        shutil.copy(f, out / f.name)
    print('Saved to Drive:', out)
    !ls -la {out}


## Summary

- All four ablation stages trained **in Colab on a T4 GPU** — logs, curves, and
  the tuned-threshold ablation table are reproducible by anyone with the repo
  and the Drive data folder.
- Checkpoints + metrics are archived to Drive under `colab_runs/<timestamp>/`.
- The saved checkpoints are the exact artifacts the FastAPI service
  (`scripts/serve_api.py`) loads for the live demo.
